# Example notebook for Ensemble of sea ice maps


##### This notebook shows how `EnsembleMaps` can be used for the monthly sea ice climatologies. The files hold one variable per hemisphere, so each hemisphere is processed on its own.


In [ ]:
import glob

import xarray as xr

from aqua.diagnostics import EnsembleMaps, PlotEnsembleMaps, merge_from_data_files

##### Location of the output of each realization


In [ ]:
# Folder with the output of `aqua analysis` for each realization: <outputdir>/<catalog>/<model>/<exp>
exp_dir = "/path/to/outputdir/climatedt-gen2/IFS-NEMO-10km/baseline-hist"
prefix = "climatedt-gen2.IFS-NEMO-10km.baseline-hist"
realizations = ["r2", "r3", "r4"]
names = {
    "catalog_list": "climatedt-gen2",
    "model_list": "IFS-NEMO-10km",
    "exp_list": "baseline-hist",
    "source_list": "lra-r100-monthly",
}
outputdir = "./output"

##### The source of the realizations is part of the filename, so it is matched with a wildcard.


In [ ]:
variable = "sea_ice_fraction_arctic"
file_list = [
    glob.glob(f"{exp_dir}/{r}/seaice/netcdf/seaice.bias.{prefix}.{r}.fraction.*.arctic_antarctic.nc")[0] for r in realizations
]
dataset = merge_from_data_files(variable=variable, data_path_list=file_list)
dataset

##### The statistics keep the `month` dimension.


In [ ]:
ens = EnsembleMaps(
    var=variable,
    dataset=dataset,
    diagnostic_product="EnsembleSeaIceMaps",
    region="arctic",
    outputdir=outputdir,
    **names,
)
ens.run()
ens.dataset_mean

##### Ensemble mean in March and September


In [ ]:
map_args = {
    "var": variable,
    "dim": "month",
    "panels": [3, 9],
    "titles": ["March", "September"],
    "proj": "nh_polar_stereo",
    "extent": [-180, 180, 50, 90],
    "figsize": (10, 5),
    "save_format": "png",
}

plot = PlotEnsembleMaps(diagnostic_product="EnsembleSeaIceMaps", region="arctic", outputdir=outputdir, **names)
fig = plot.plot_seasonal(dataset=ens.dataset_mean, vmin=0, vmax=1, cmap="Blues_r", data_name="ensemble_mean", **map_args)

##### Bias against OSI-SAF
##### The reference covers one hemisphere only: the difference uses the latitudes it shares with the model.


In [ ]:
ref_file = f"{exp_dir}/r2/seaice/netcdf/seaice.bias.obs.OSI-SAF.osi-saf-ssmis.r1.fraction.nh-monthly.arctic.nc"
reference = xr.open_dataset(ref_file)[variable]

fig = plot.plot_seasonal(
    dataset=ens.dataset_mean, ref_dataset=reference, vmin=-0.5, vmax=0.5, data_name="ensemble_bias_mean", **map_args
)